# Hierarchical classifier: DeBERTa-v3-base, k = 0, 3 seeds

Stage 1: `other` vs `not_other` (all segments). Stage 2: the 6 target classes (trained on non-`other` segments). Combined: p(other) = p1(other), p(c) = p1(not_other) × p2(c); label = highest combined score.

Temporary trial (`scripts/trial_run.py --mode hierarchical`); its numbers are **not reportable** (PROTOCOL §4.1).

Before running, in the notebook settings:
- **Accelerator:** GPU T4 (one GPU is used; fp16 + fused AdamW by default, `--fp32` for the pre-optimization setup)
- **Internet:** on (to clone the repo and download the model)
- **Input:** the private `smi-v4-meetings` dataset attached

The repo branch `C/baselines-design` must be pushed: the notebook clones it from GitHub.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
Do **not** install `requirements.txt` here: it would replace Kaggle's own torch build.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C/baselines-design https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 sentencepiece==0.2.2

## 3. Data folder
Found automatically: the folder that contains `source.json`, `train/`, `val/`, `test/`.

In [ ]:
import glob, os

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
print("source.json found at:", hits)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))

## 4. Model loads in fp32
Also downloads the weights once (371 MB).

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo/src")
from smi.models.model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer("microsoft/deberta-v3-base")
print("dtype:", model.dtype)  # must be torch.float32
assert str(model.dtype) == "torch.float32"
del model

## 5. Run: 3 seeds, one GPU, fp16 + fused AdamW
Deletes old trial runs first (they are throwaway; remove the `rm` line once you have runs to keep).

In [ ]:
!rm -rf /kaggle/working/runs
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!cd /kaggle/working/repo && PYTHONPATH=src python scripts/trial_run.py --mode hierarchical --seeds 42 43 44 \
    --data-root {DATA_ROOT} --model microsoft/deberta-v3-base --out-dir /kaggle/working/runs

## 6. Results

In [ ]:
import json, statistics

scores = []
for run in sorted(glob.glob("/kaggle/working/runs/*")):
    print(f"\n=================== {os.path.basename(run)}")
    for name in ("metrics_stage1.json", "metrics_stage2.json"):
        r = json.load(open(f"{run}/{name}"))
        print(f"--- {name} (alone) | selected epoch {r['selected_epoch']} | macro F1 over {r['select_labels']}: {r['macro_f1']:.4f}")
        for label, v in r["per_class"].items():
            print(f"    {label:10s} P {v['p']:.3f}  R {v['r']:.3f}  F1 {v['f1']:.3f}  support {v['support']}")
    m = json.load(open(f"{run}/metrics.json"))
    scores.append(m["primary"]["macro_f1_6"])
    print(f"--- metrics.json (both combined) | macro_f1_6 {m['primary']['macro_f1_6']:.4f} | macro_f1_7 {m['macro_f1_7']:.4f}")
    for label, v in m["per_class"].items():
        print(f"    {label:10s} P {v['p']:.3f}  R {v['r']:.3f}  F1 {v['f1']:.3f}  support {v['support']}")
    for history in sorted(glob.glob(f"{run}/*/history.jsonl")):
        stage = os.path.basename(os.path.dirname(history))
        for line in open(history):
            h = json.loads(line)
            if h["type"] == "epoch":
                print(f"  [{stage}] epoch {h['epoch']}: train_loss {h['train_loss']:.4f} | val macro F1 {h['val_macro_f1']:.4f} "
                      f"| best {h['is_best']} | {h['seconds'] / 60:.1f} min")

print("\ncombined macro_f1_6 per seed:", [round(s, 4) for s in scores])
if len(scores) > 1:
    print(f"mean {statistics.mean(scores):.4f} , std {statistics.stdev(scores):.4f}")